# Lunar Crater Detection and Shapefile Generation Pipeline

This notebook demonstrates the end-to-end workflow of georeferencing Orbiter High Resolution Camera (OHRC) imagery from Chandrayaan-2, generating tiles, running inference using a trained YOLOv8 model, and producing a validated GIS-ready Shapefile of detected craters.

## Step 1: Georeferencing using GCPs

We read the ground control points (GCPs) from the geometry CSV file, build a virtual raster (VRT), and warp the raw TIFF file (`ohrc_image.tif`) into a georeferenced GeoTIFF (`ohrc_georeferenced.tif`) with projection `EPSG:4326` (lunar coordinates mapping to geographic decimal degrees).

In [ ]:
import os
from src.georeference import georeference_image

# Run georeferencing
georeference_image(
    tif_path="ohrc_image.tif",
    csv_path="data/Geometry/20190907/ch2_ohr_ncp_20190907T0438126359_g_grd_g26.csv",
    vrt_path="ohrc_with_gcps.vrt",
    geotiff_path="ohrc_georeferenced.tif"
)

## Step 2: Image Tiling

We slice the georeferenced GeoTIFF into `512x512` tiles, saving them as PNGs and creating a metadata file `tile_metadata.json` that stores the exact geotransform matrix for each window segment.

In [ ]:
from src.tiling import tile_image

# Slice the GeoTIFF into tiles
tile_image(
    input_tif="ohrc_georeferenced.tif",
    tile_size=512,
    output_dir="Tiles"
)

## Step 3: Run YOLOv8 Inference

We load the trained model `yolov8Crater.pt` and perform inference on the generated tiles, writing normalized bounding boxes to `yolo_predictions/`.

In [ ]:
from src.inference import run_inference

# Run inference on the sliced png tiles
run_inference(
    model_path="yolov8Crater.pt",
    image_dir="Tiles",
    output_dir="yolo_predictions",
    conf=0.25
)

## Step 4: Convert Bounding Boxes to Selenographic Shapefile

Using the corrected mapping code (`Affine.from_gdal`), we project the pixel-level bounding boxes back to absolute lunar degrees and write a GIS-compatible Shapefile (`detected_boxes.shp`).

In [ ]:
from src.bbox_to_shp import generate_shapefile

# Generate fixed coordinate shapefile
generate_shapefile(
    tile_dir="Tiles",
    pred_dir="yolo_predictions",
    output_shapefile="detected_boxes"
)

## Step 5: Verification of Coordinates

We inspect the shapefile bounds using the `shapefile` library to verify that they are correctly mapped to selenographic decimal degrees (within valid spherical coordinate limits) and not scrambled.

In [ ]:
import shapefile

sf = shapefile.Reader("detected_boxes.shp")
print("Total shapes detected:", len(sf.shapes()))
if len(sf.shapes()) > 0:
    print("First shape bounding box (Lon/Lat bounds):", sf.shapes()[0].bbox)
    print("First shape record attributes:", sf.record(0))